# ReelSmith - Colab runner

Runs the reel pipeline **inside Google Colab**, against footage that already
lives in your Drive. Nothing is downloaded to your laptop.

Two jobs:

1. **Proxies and transcript.** The rushes are hundreds of megabytes each, more
   than a chat upload or the Drive connector will carry. Cells 3 and 4 write small
   proxies and a Whisper transcript back to Drive, and those *are* small enough to
   hand to Claude.
2. **Full-resolution work.** Cells 5 and 6 run shot detection and reframing
   against the original files and write the results to Drive.

Runtime -> Change runtime type -> **T4 GPU** makes Whisper roughly ten times
faster. Everything else here is CPU-bound and fine on the free tier.

## 1. Mount Drive and install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null
!pip -q install opencv-python-headless==4.10.0.84 scenedetect opentimelineio faster-whisper
!git clone -q https://github.com/foonkoons-cyber/oliv.git /content/oliv || true
!cd /content/oliv && git checkout -q claude/online-video-editing-xj0vuz && git pull -q
import sys; sys.path.insert(0, '/content/oliv/reelsmith')
print('ready')

## 2. Point at the footage

`SRC` is the Drive folder holding the rushes. The first cell prints the folders
under *My Drive* so the name can be copied rather than guessed.

If the folder was shared with you rather than owned by you, open it in Drive once
and use **Add shortcut to Drive** first — mounted Drive only exposes what is
reachable from *My Drive*.


In [ ]:
import os, glob, subprocess, json, pathlib

for d in sorted(glob.glob('/content/drive/MyDrive/*')):
    if os.path.isdir(d):
        print(os.path.basename(d))


In [ ]:
SRC   = '/content/drive/MyDrive/PASTE THE FOLDER NAME FROM ABOVE'
WORK  = '/content/drive/MyDrive/reelsmith'
PROXY = f'{WORK}/proxies'
OUT   = f'{WORK}/out'
for d in (PROXY, OUT):
    os.makedirs(d, exist_ok=True)

VIDEO_EXT = ('.mov', '.mp4', '.mxf', '.m4v', '.avi')
files = sorted(f for f in glob.glob(f'{SRC}/*') if f.lower().endswith(VIDEO_EXT))
assert files, f'nothing found in {SRC}'

def role(path):
    n = os.path.basename(path).lower()
    if 'interview' in n:  return 'interview'
    if 'drone' in n:      return 'drone'
    return 'broll'

total = sum(os.path.getsize(f) for f in files)
print(f'{len(files)} files, {total/1e9:.2f} GB')
for f in files:
    print(f'  {os.path.getsize(f)/1e6:8.1f} MB  {role(f):9s}  {os.path.basename(f)}')


## 3. Manifest

Probes every file and writes `manifest.json`. A few kilobytes, so it can be read
straight away — duration, frame rate and resolution are enough to start planning
which clips can carry slow motion and how much footage there is to cut from,
before any proxy has finished encoding.


In [ ]:
def probe(path):
    out = subprocess.run(
        ['ffprobe', '-v', 'error', '-select_streams', 'v:0', '-show_entries',
         'stream=width,height,r_frame_rate,codec_name:format=duration',
         '-of', 'json', path], capture_output=True, text=True).stdout
    d = json.loads(out)
    s = d['streams'][0]
    num, den = (float(v) for v in s['r_frame_rate'].split('/'))
    return {'width': s['width'], 'height': s['height'],
            'fps': round(num / den if den else num, 3),
            'codec': s.get('codec_name'),
            'duration': round(float(d['format']['duration']), 2)}

manifest = []
for f in files:
    info = probe(f)
    info.update(file=os.path.basename(f), role=role(f),
                size_mb=round(os.path.getsize(f) / 1e6, 1))
    manifest.append(info)
    print(f"  {info['duration']:7.2f}s  {info['width']}x{info['height']}"
          f"  {info['fps']:6.2f}fps  {info['role']:9s}  {info['file']}")

json.dump(manifest, open(f'{WORK}/manifest.json', 'w'), indent=1)
secs = sum(m['duration'] for m in manifest)
print(f"\n{len(manifest)} files, {secs/60:.1f} min of footage -> manifest.json")


## 4. Proxies, small enough to hand over

540p for b-roll and the drone plate, 480p for the interview, plus an audio-only
pull. The target is under 10 MB per file, which is the Drive connector's per-file
ceiling — if a proxy overshoots, the loop raises the CRF and re-encodes. Files
that already have a proxy under the ceiling are skipped, so a re-run after an
interrupted session is cheap.

About three minutes per gigabyte on a free Colab CPU, so budget roughly ten
minutes for a 2.8 GB set.


In [ ]:
LIMIT = 9.5e6          # the Drive connector's per-file ceiling, with headroom

def make_proxy(src, dst, height, crf, audio):
    """Encode until it fits under LIMIT. Frame rate is left alone, so a 60p
    rush stays 60p and slow motion is still available later."""
    for attempt in (crf, crf + 4, crf + 8, crf + 12):
        cmd = ['ffmpeg', '-v', 'error', '-y', '-i', src,
               '-vf', f'scale=-2:{height}',
               '-c:v', 'libx264', '-crf', str(attempt), '-preset', 'veryfast']
        cmd += ['-c:a', 'aac', '-b:a', '64k'] if audio else ['-an']
        subprocess.run(cmd + [dst], check=True)
        if os.path.getsize(dst) <= LIMIT:
            return attempt
    return None

for k, f in enumerate(files, 1):
    base = pathlib.Path(f).stem
    r = role(f)
    dst = f'{PROXY}/{base}__proxy.mp4'
    if os.path.exists(dst) and os.path.getsize(dst) <= LIMIT:
        print(f'[{k}/{len(files)}] have it already: {base}')
    else:
        used = make_proxy(f, dst, 480 if r == 'interview' else 540,
                          32 if r == 'interview' else 31, r == 'interview')
        print(f'[{k}/{len(files)}] {os.path.getsize(dst)/1e6:5.1f} MB  crf {used}'
              f'  {base}' + ('' if used else '   << still over, drop the height'))
    if r == 'interview':
        aud = f'{PROXY}/{base}__audio.m4a'
        if not os.path.exists(aud):
            subprocess.run(['ffmpeg', '-v', 'error', '-y', '-i', f, '-vn',
                            '-ac', '1', '-c:a', 'aac', '-b:a', '96k', aud],
                           check=True)
        print(f'          {os.path.getsize(aud)/1e6:5.1f} MB  {base}__audio.m4a')


## 5. Transcript with word-level timing

The one step that cannot run in the Claude session - the model hosts are blocked
there - so it runs here and the result goes to Drive as JSON and SRT. Word timings
are what let captions land on the syllable and b-roll cut on the beat of a
sentence.

In [ ]:
from faster_whisper import WhisperModel
import torch

audio = sorted(glob.glob(f'{PROXY}/*__audio.m4a'))
assert audio, 'run cell 3 first'
gpu = torch.cuda.is_available()
model = WhisperModel('large-v3' if gpu else 'base.en',
                     device='cuda' if gpu else 'cpu',
                     compute_type='float16' if gpu else 'int8')

def srt_time(t):
    h, m, s = int(t // 3600), int(t % 3600 // 60), t % 60
    return f'{h:02d}:{m:02d}:{s:06.3f}'.replace('.', ',')

for a in audio:
    segs, info = model.transcribe(a, word_timestamps=True, vad_filter=True,
                                  beam_size=5, condition_on_previous_text=False)
    words, lines = [], []
    for i, s in enumerate(segs, 1):
        stamp = srt_time(s.start) + ' --> ' + srt_time(s.end)
        lines.append(str(i) + chr(10) + stamp + chr(10) + s.text.strip() + chr(10))
        for w in s.words or []:
            if w.word.strip():
                words.append({'t': w.word.strip(),
                              's': round(w.start, 3), 'e': round(w.end, 3)})
    stem = pathlib.Path(a).stem.replace('__audio', '')
    json.dump(words, open(f'{PROXY}/{stem}__words.json', 'w'), indent=0)
    open(f'{PROXY}/{stem}.srt', 'w').write(chr(10).join(lines))
    print(f'{stem}: {len(words)} words -> {stem}__words.json + .srt')


## 6. Shot detection and scoring

Run this on the **originals**, not the proxies - boundaries and sharpness should
be measured on the real thing. The resulting `shots.json` is tiny.

In [ ]:
brolls = [f for f in files if role(f) != 'interview']
args = ' '.join("'" + b + "'" for b in brolls)
!cd /content/oliv/reelsmith && python3 shots.py {args} --out '{WORK}/shots.json' --thumbs '{WORK}/thumbs'

## 7. Reframe 16:9 to 9:16, keeping the subject in frame

`--mode hybrid` crops and tracks where the subject fits and falls back to a
blurred fill where they do not, so nobody is ever cropped out. `--qc` drops a
contact sheet beside each render so a whole shot can be checked at a glance.

In [ ]:
shots = json.load(open(f'{WORK}/shots.json'))['shots']
usable = [s for s in shots if s['usable'] and s['people_rate'] >= 0.3]
print(f'{len(usable)} shots with a person in them')

# The interview is the reference for who the subject is. Spread the sample
# times across it -- one frame's signature does not survive a lighting change.
interview = next(f for f in files if role(f) == 'interview')
idur = next(m['duration'] for m in manifest if m['role'] == 'interview')
ref_at = ','.join(f'{idur * p:.1f}' for p in (0.08, 0.25, 0.45, 0.65, 0.85))
print(f'reference: {os.path.basename(interview)} @ {ref_at}s')

os.makedirs(f'{OUT}/shots', exist_ok=True)
for s in usable:
    dst = f"{OUT}/shots/{s['id']}.mp4"
    subprocess.run(
        ['python3', 'reframe.py', '--input', s['file'], '--out', dst,
         '--start', str(s['start']), '--end', str(s['end']),
         '--size', '1080x1920', '--mode', 'hybrid',
         '--subject', 'match', '--ref-clip', interview, '--ref-at', ref_at,
         '--report', dst.replace('.mp4', '.json'),
         '--qc', dst.replace('.mp4', '_qc.png')],
        cwd='/content/oliv/reelsmith', check=True)


## 8. Hand back to Claude

Share `MyDrive/reelsmith` and say so in the chat. What gets read:

| File | Why |
|---|---|
| `manifest.json` | every clip's duration, fps and resolution |
| `proxies/*__proxy.mp4` | to look at the footage and judge the shots |
| `proxies/*__words.json` | word timing — hook selection and caption sync |
| `shots.json` | the shot list with scores |
| `thumbs/*.jpg` | one frame per shot, for picking b-roll |
| `out/shots/*_qc.png` | proof that nobody left the frame |

All of those sit well under the 10 MB per-file ceiling. The rushes stay in Drive.
